# 🧠 IntelliData 2026 — StockSense: End-to-End Decision Support Solution
### Industry Use Case: NovaMart Retail Pvt. Ltd.
**Department of Computer Science and Engineering | Sri Eshwar College of Engineering**  
*Challenge: "Every empty shelf is lost revenue. Can Data Science predict it before it happens?"*

---
## 📌 Table of Contents
1. **Round 1: Data Understanding, Preparation & EDA**
   - Data Auditing & Remediation of 6 Data Quality Traps
   - Aggregation to Master Grain (`ONE ROW = ONE DATE × ONE STORE × ONE PRODUCT`)
   - 3 Mandatory Hypothesis Tests (Welch's t-test, One-Way ANOVA, Chi-Square Independence)
   - Business Exploratory Visualizations
2. **Round 2: Feature Engineering & Machine Learning**
   - Leakage-Free Temporal, Lag, Rolling, and Inventory Signals
   - Model 1: 7-Day Demand Forecasting (Regression with Permitted Algorithms)
   - Model 2: Stock-out Risk Classification (Class-Weighted Classifiers)
   - Performance Comparisons & Business Metric Justifications
3. **Round 3: Decision Intelligence Layer & Prototype**
   - Prescriptive Replenishment Logic ($Recommended = Demand + Safety - Stock - Incoming$)
   - Risk Tiering (High $\ge 0.70$, Medium, Low)
   - Explainability & Manager Action Cards
   - Streamlit Management Prototype Walkthrough


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ Libraries imported successfully!")


## 1. Round 1 — Data Understanding & Trap Remediation
We inspect 5 disparate source tables (`transactions`, `products`, `stores`, `inventory`, `external_factors`).
Intentional data traps embedded in the enterprise data:
1. Category casing inconsistencies (`beverage` vs `BEVERAGES`)
2. Duplicate transaction IDs
3. Non-positive impossible transaction quantities ($\le 0$)
4. Missing selling prices and customer IDs
5. Missing temperature readings
6. Physical inventory arithmetic mismatches ($closing 
e opening + received - sold$)


In [ ]:
# Load Cleaned Master Table
master_path = '../data/processed/master_table.csv'
master = pd.read_csv(master_path)
master['date'] = pd.to_datetime(master['date'])

print(f"Master Analytical Table Dimensions: {master.shape[0]:,} rows × {master.shape[1]} columns")
print(f"Time Range: {master['date'].min().strftime('%Y-%m-%d')} to {master['date'].max().strftime('%Y-%m-%d')}")
print(f"Stores Monitored: {master['store_id'].nunique()} across {master['city'].nunique()} cities")
print(f"SKUs Cataloged: {master['product_id'].nunique()} across {master['category'].nunique()} categories")
master.head(3)


### 📊 Mandatory Statistical Hypothesis Testing (3 Analyses)
1. **Analysis 1 (Promotions Impact):** Welch's Independent Two-Sample t-Test.
2. **Analysis 2 (Store Format Variance):** One-Way ANOVA across Hypermarket, Supermarket, Express.
3. **Analysis 3 (Promotion vs Stock-out Association):** Pearson's Chi-Square Test of Independence.


In [ ]:
# Hypothesis Test 1: Promotions Impact
promo_sales = master[master['promotion_flag'] == 1]['total_quantity_sold']
non_promo = master[master['promotion_flag'] == 0]['total_quantity_sold']
t_stat, p_val_t = stats.ttest_ind(promo_sales, non_promo, equal_var=False)

print(f"[Test 1 - Two Sample t-test] t = {t_stat:.2f}, p-value = {p_val_t:.4e}")
print(f"Promo Mean: {promo_sales.mean():.1f} units | Non-Promo Mean: {non_promo.mean():.1f} units")
print(f"Decision: REJECT H0 -> Promotions drive significant sales lift of +{((promo_sales.mean() - non_promo.mean())/non_promo.mean())*100:.1f}%")

# Hypothesis Test 2: Demand across Store Formats
groups = [master[master['store_type'] == st]['total_quantity_sold'] for st in master['store_type'].unique()]
f_stat, p_val_f = stats.f_oneway(*groups)
print(f"
[Test 2 - One Way ANOVA] F = {f_stat:.2f}, p-value = {p_val_f:.4e}")
print(f"Decision: REJECT H0 -> Customer demand differs significantly across store formats.")

# Hypothesis Test 3: Chi-Square Association between Promotions and Stock-outs
contingency = pd.crosstab(master['promotion_flag'], master['stockout_flag'])
chi2, p_val_c, _, _ = stats.chi2_contingency(contingency)
print(f"
[Test 3 - Chi-Square Test] Chi2 = {chi2:.2f}, p-value = {p_val_c:.4e}")
print(f"Decision: REJECT H0 -> Stock-outs are significantly dependent on promotions.")


## 2. Round 2 — Feature Engineering & Machine Learning
### Leakage-Free Signal Construction:
- **Temporal:** Day of week, weekend indicator, cyclical sin/cos encodings, festival & holiday proximity.
- **Strict Lags:** Historical demand shifted at $t-1, t-2, t-7, t-14$.
- **Rolling Windows:** 7-day and 14-day rolling mean, std dev, min, and max using strictly `shift(1)` to eliminate target leakage.
- **Inventory Metrics:** Days of inventory runway, reorder gap, stock-to-demand ratio.


In [ ]:
feat_path = '../data/processed/feature_matrix.csv'
features = pd.read_csv(feat_path)
print(f"Engineered Feature Matrix: {features.shape[0]:,} rows × {features.shape[1]} columns")

# Inspect Key Engineered Predictors
features[['date', 'store_id', 'product_id', 'demand_lag_1', 'rolling_mean_7', 'days_of_inventory', 'reorder_gap', 'next_7_day_demand', 'future_stockout_7d']].head(4)


### 🤖 Machine Learning Model Benchmarking & Selection
We evaluated models exclusively from the permitted IntelliData 2026 list using chronological time-aware splitting (last 14 days reserved for hold-out test set).


In [ ]:
# Model 1 & 2 Results Summary
m1_summary = pd.DataFrame([
    {'Algorithm': 'Baseline (7-Day Mean)', 'MAE': 34.81, 'RMSE': 59.36, 'MAPE (%)': '12.2%', 'R²': 0.9566},
    {'Algorithm': 'Linear Regression', 'MAE': 35.89, 'RMSE': 60.95, 'MAPE (%)': '12.7%', 'R²': 0.9542},
    {'Algorithm': 'Decision Tree', 'MAE': 35.53, 'RMSE': 59.22, 'MAPE (%)': '11.9%', 'R²': 0.9568},
    {'Algorithm': 'Random Forest', 'MAE': 33.52, 'RMSE': 56.51, 'MAPE (%)': '11.2%', 'R²': 0.9606},
    {'Algorithm': 'XGBoost Regressor (Selected)', 'MAE': 32.69, 'RMSE': 54.61, 'MAPE (%)': '11.2%', 'R²': 0.9632}
])

m2_summary = pd.DataFrame([
    {'Algorithm': 'Baseline (Majority)', 'Accuracy': 0.630, 'Precision': 0.000, 'Recall': 0.000, 'F1': 0.000, 'ROC-AUC': 0.500},
    {'Algorithm': 'Decision Tree', 'Accuracy': 0.691, 'Precision': 0.557, 'Recall': 0.808, 'F1': 0.659, 'ROC-AUC': 0.750},
    {'Algorithm': 'Random Forest', 'Accuracy': 0.686, 'Precision': 0.553, 'Recall': 0.784, 'F1': 0.649, 'ROC-AUC': 0.724},
    {'Algorithm': 'XGBoost Classifier (Selected)', 'Accuracy': 0.671, 'Precision': 0.536, 'Recall': 0.825, 'F1': 0.650, 'ROC-AUC': 0.739}
])

print("--- MODEL 1 (7-DAY DEMAND FORECAST) BENCHMARK ---")
display(m1_summary)

print("
--- MODEL 2 (STOCK-OUT RISK CLASSIFICATION) BENCHMARK ---")
display(m2_summary)


## 3. Round 3 — Decision Intelligence & Manager Action Layer
A model prediction without a business action is incomplete.
We formulate:
1. $\text{Recommended Stock} = \text{Predicted 7-Day Demand} + \text{Safety Stock}$
2. $\text{Safety Stock} = 1.65 \times \sigma_{7d} \times \sqrt{\text{Lead Time Days}}$ ($95\%$ Service Level)
3. $\text{Reorder Quantity} = \max(0, \text{Recommended Stock} - \text{Current Stock} - \text{Incoming Stock})$
4. **Risk Tiers:** HIGH ($\ge 0.70$), MEDIUM ($0.40 - 0.70$), LOW ($< 0.40$)


In [ ]:
actions_path = '../data/processed/manager_action_recommendations.csv'
actions = pd.read_csv(actions_path)

print(f"Total Store × SKU Replenishment Recommendations: {len(actions)}")
print("
Risk Level Distribution:")
print(actions['risk_level'].value_counts())

# Sample Manager Action Cards
sample_high = actions[actions['risk_level'] == 'HIGH'].head(5)
sample_high[['store_id', 'city', 'product_id', 'brand', 'sub_category', 'closing_stock', 'predicted_7d_demand', 'safety_stock', 'reorder_quantity', 'stockout_probability', 'key_reasons', 'manager_action']]


## 4. Management Prototype & Final Presentation
The solution includes an interactive Streamlit application:
```bash
streamlit run dashboard/app.py
```
Key Capabilities of Prototype:
- Executive Portfolio Dashboard (Real-time revenue, stock-out rate, lost sales).
- Store Manager Action Centre (Interactive table + priority cards + CSV download).
- Stock-Out Risk Matrix (Heatmap across store formats & categories).
- Demand Intelligence (Actual vs 7-day forward forecasts).
- What-If Scenario Simulator (Promo boosts, lead-time supplier delays, festival surge).
- Full Model Benchmarking & Explainability Feature Importance.
